# AI 教学工作台（MVP v0.1）\n读取 Google Drive 任务单，调用 Gemini API，生成教学设计、逐字稿、HTML 课件、图片提示词和 SVG 视觉卡片。\n\n**先检查任务 JSON，再运行生成单元格。不要把 API key 写进 Notebook 或任务文件。**

In [ ]:
%pip -q install -U google-genai

In [ ]:
from google.colab import drive, userdata\nfrom pathlib import Path\nimport json, os, re, html, time, getpass\nfrom datetime import datetime\n\ndrive.mount('/content/drive')\nBASE = Path('/content/drive/MyDrive/AI教学工作台')\nTASK_PATH = BASE / 'tasks' / 'task.json'\nOUTPUTS = BASE / 'outputs'\nBASE.joinpath('tasks').mkdir(parents=True, exist_ok=True)\nOUTPUTS.mkdir(parents=True, exist_ok=True)\n\nif not TASK_PATH.exists():\n    print('未找到任务文件。请把仓库中的 task_template.json 上传为：', TASK_PATH)\n    raise FileNotFoundError(TASK_PATH)\ntask = json.loads(TASK_PATH.read_text(encoding='utf-8'))\nrequired = ['task_id', 'lesson_title', 'grade', 'lesson_count']\nmissing = [k for k in required if not task.get(k)]\nif missing: raise ValueError(f'任务单缺少必填字段: {missing}')\nprint('已读取任务：', task['task_id'], '|', task['lesson_title'], '|', task['grade'])\nprint('请核对课文、年级、课时与额外要求；确认无误后再运行下一单元格。')

In [ ]:
# 人工确认闸门：输入 YES 才会调用模型并产生文件\napproval = input('我已检查任务 JSON，确认开始生成请输入 YES： ').strip()\nif approval != 'YES':\n    raise RuntimeError('已停止：教师未确认。本单元格不会调用模型。')\n\ntry:\n    API_KEY = userdata.get('GEMINI_API_KEY')\nexcept Exception:\n    API_KEY = None\nif not API_KEY:\n    API_KEY = getpass.getpass('输入 Gemini API key（仅当前运行使用，不会保存到文件）： ')\nif not API_KEY.strip(): raise ValueError('没有提供 API key')\n\nfrom google import genai\nfrom google.genai import types\nMODEL = os.environ.get('GEMINI_MODEL', 'gemini-2.5-flash')\nclient = genai.Client(api_key=API_KEY)\nprint('确认通过；模型配置：', MODEL)

In [ ]:
prompt = f'''你是一支小学语文备课团队。请依据以下任务生成可供教师审核的中文教学材料。\n任务 JSON：\n{json.dumps(task, ensure_ascii=False, indent=2)}\n\n请只返回合法 JSON 对象，不要 Markdown 代码围栏，字段必须包括：\nteaching_design：字符串，包含教材分析、学情分析、教学目标、重点难点、课前准备、两课时流程表（时间/教师活动/学生活动/设计意图/评价）、板书设计、作业与反思点；\nteacher_script：字符串，按课时分段，含教师语言、预设学生回答、追问、过渡语、朗读指导、课堂评价；\nslides：数组，每页含 title、bullets（字符串数组）、speaker_notes、image_prompt；建议 8-14 页；\nimage_prompts：数组，提供适合小学课堂的配套插图提示词，注明主体、场景、构图、风格、横竖版及避免的错误；\nquality_checks：数组，列出教师需要核实的教材事实、版本和教学设计风险。\n\n不要编造教材原文、页码、名师案例或参考文献。缺少依据时明确写“需教师核对”。内容适合小学四年级，活动、目标和评价相互对应。'''\n\nresponse = client.models.generate_content(\n    model=MODEL,\n    contents=prompt,\n    config=types.GenerateContentConfig(response_mime_type='application/json', temperature=0.4)\n)\nraw = (response.text or '').strip()\nif raw.startswith('```'):\n    raw = re.sub(r'^```(?:json)?\\s*|\\s*```$', '', raw, flags=re.I)\ndata = json.loads(raw)\nfor key in ['teaching_design','teacher_script','slides','image_prompts','quality_checks']:\n    if key not in data: raise ValueError(f'模型结果缺少字段: {key}')\nprint(f'生成完成：{len(data[\"slides\"])} 页课件草稿；请检查输出，不要直接视为定稿。')

In [ ]:
# 导出成果：每个任务使用独立目录，避免覆盖其他课文\ntask_id = re.sub(r'[^0-9A-Za-z_一-龥-]+', '-', str(task['task_id'])).strip('-') or datetime.now().strftime('%Y%m%d-%H%M%S')\nout_dir = OUTPUTS / task_id\nout_dir.mkdir(parents=True, exist_ok=True)\n(out_dir / '教学设计.md').write_text(data['teaching_design'], encoding='utf-8')\n(out_dir / '课堂逐字稿.md').write_text(data['teacher_script'], encoding='utf-8')\n(out_dir / '图片提示词.md').write_text('\\n\\n'.join(str(x) for x in data['image_prompts']), encoding='utf-8')\n(out_dir / '教师核验清单.md').write_text('\\n'.join(f'- [ ] {x}' for x in data['quality_checks']), encoding='utf-8')\n\ndef svg_card(title, subtitle, idx):\n    title = html.escape(str(title)[:45]); subtitle = html.escape(str(subtitle)[:80])\n    return f'''<svg xmlns="http://www.w3.org/2000/svg" width="1280" height="720" viewBox="0 0 1280 720"><rect width="1280" height="720" fill="#f5f3ee"/><circle cx="1080" cy="130" r="170" fill="#d8e7e1"/><circle cx="1120" cy="620" r="220" fill="#ead7bd"/><rect x="80" y="90" width="1120" height="540" rx="32" fill="#ffffff" stroke="#c7d4cf" stroke-width="3"/><text x="130" y="180" font-size="30" fill="#52665e">AI 教学工作台 · 视觉卡片 {idx:02d}</text><text x="130" y="330" font-size="54" font-weight="700" fill="#203c34">{title}</text><foreignObject x="130" y="385" width="980" height="150"><div xmlns="http://www.w3.org/1999/xhtml" style="font:28px sans-serif;color:#52665e;line-height:1.5">{subtitle}</div></foreignObject></svg>'''\n\nslides = data['slides']\nslide_html = []\nfor i, s in enumerate(slides, 1):\n    title = html.escape(str(s.get('title','未命名页面')))\n    bullets = ''.join(f'<li>{html.escape(str(b))}</li>' for b in s.get('bullets', []))\n    notes = html.escape(str(s.get('speaker_notes','')))\n    slide_html.append(f'<section class="slide"><small>第 {i} 页</small><h1>{title}</h1><ul>{bullets}</ul><details><summary>教师备注</summary><p>{notes}</p></details></section>')\n    (out_dir / f'视觉卡片-{i:02d}.svg').write_text(svg_card(s.get('title',''), '；'.join(map(str,s.get('bullets',[])[:2])), i), encoding='utf-8')\ndeck = '''<!doctype html><html lang="zh-CN"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>教学课件草稿</title><style>body{margin:0;background:#e9ece8;color:#23352e;font-family:system-ui,-apple-system,sans-serif}.slide{box-sizing:border-box;background:#fff;margin:28px auto;padding:5vw;max-width:1100px;min-height:560px;border-radius:18px;box-shadow:0 8px 28px #0001}.slide h1{font-size:clamp(28px,4vw,52px)}li{font-size:clamp(18px,2vw,28px);line-height:1.7;margin:.5em 0}small,summary{color:#5d766b}details{margin-top:2em}</style></head><body>''' + ''.join(slide_html) + '</body></html>'\n(out_dir / '课件草稿.html').write_text(deck, encoding='utf-8')\n(out_dir / '模型原始结果.json').write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')\nlog = {'task_id': task_id, 'status': 'success', 'model': MODEL, 'finished_at': datetime.now().isoformat(timespec='seconds'), 'files': sorted(p.name for p in out_dir.iterdir())}\n(out_dir / 'run_log.json').write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding='utf-8')\nprint('成果已写入：', out_dir)\nfor p in sorted(out_dir.iterdir()): print('-', p.name)

## 人工验收后再使用\n1. 打开 `教学设计.md` 和 `课堂逐字稿.md`，核对教材原文、目标、时间和师生对话。\n2. 打开 `课件草稿.html` 检查投影效果；它是可浏览器打开的 HTML 草稿，不是 PPTX。\n3. `图片提示词.md` 可用于后续已授权的图片生成工具；SVG 视觉卡片是简易排版素材，不是真实场景插画。\n4. 如 API 报错，先检查 API key、账号模型权限、额度和模型名称；不要把密钥或完整报错中的敏感信息发到公开仓库。